# 03: 标准化 + log1p + 高可变基因 HVG

在 02 过滤后的 counts 矩阵上执行标准化管线，为后续 04 降维与 05 聚类准备数据。

**为什么需要这一步？** 不同细胞的测序深度天然不同（有的被测了 5,000 条 RNA，
有的 50,000 条）。如果不做标准化，高深度细胞会主导后续所有分析。
此外，单细胞计数数据高度离散、严重右偏，不适合 PCA 等线性降维方法——
log1p 将其拉近正态分布。HVG 选择则进一步滤掉"在所有细胞中表达量都差不多"的管家基因，
只保留携带细胞类型差异信息的高可变基因，大幅降噪并节省内存。

**本 notebook 产出**：
- `adata.layers['counts']` — 原始 counts，为下游 scVI / scANVI / DESeq2 保留
- `adata.X` — 标准化 + log1p 变换后的 float32
- `adata.var['highly_variable']` — HVG 布尔掩码
- `adata.uns['normalize_v1']` — 标准化参数记录
- 03 checkpoint `.h5ad` 文件，供 04 嵌入使用

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：02（QC + 过滤），读 `02_qcd_v*.h5ad`
- **下游**：04（多方法嵌入），产出 `03_normalized_v*.h5ad`

### 什么时候需要回跑？
在后续 04（看 UMAP 嵌入效果）、05（看 Leiden 分群是否合理）、
06（细胞注释时发现误分群）的过程中，都可能发现需要回到这里调整 HVG 数量或方法。

### 操作步骤
1. **改 `UPSTREAM_PATH`** — 指向要复用的上游文件版本（如 `results/02_qcd_v2.h5ad`）
2. **改 `OUTPUT_PATH`** — bump 版本号 `_v1` → `_v2`（如 `results/03_normalized_v2.h5ad`）
3. **调整参数** — 在下方 PARAMS 区域改 `N_TOP_GENES` 或 `HVG_FLAVOR`
4. **重跑** — Cell → Run All

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。旧版 `.h5ad` **不覆盖不删除**，
  保留在 `results/` 供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为参数合理、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：04 的 `UPSTREAM_PATH` 指向你决定采用的 03 版本即可。

### 追溯链（自动写入 h5ad）
本 notebook 写出前自动记录以下字段，供后续审计：
- `stage` = `"03_normalized"`
- `status` = `"experimental"`（PI 审查后手动改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

In [ ]:
# ============================================================
# PARAMS —— 运行前可调整的参数集中在这里
# ============================================================

# UPSTREAM_PATH: 02 产出文件路径。
#   如需回跑：指向你要复用的上游版本（如 02_qcd_v2.h5ad）。
UPSTREAM_PATH = "results/02_qcd_v1.h5ad"

# OUTPUT_PATH: 本 stage 产出 checkpoint 路径。
#   如需回跑：bump 版本号 _v1 → _v2，旧版不覆盖。
OUTPUT_PATH   = "results/03_normalized_v1.h5ad"

# N_TOP_GENES: 高可变基因数量（默认 2000）。
#   调大（3000-4000）= 更多基因参与降维，可能捕获更细微的生物学信号，
#   但噪声也更多、计算更慢。2000 是单细胞领域的经验默认值。
N_TOP_GENES   = 2000

# HVG_FLAVOR: 高可变基因选择算法。
#   "seurat"      — 用于 log-normalized 数据（默认，最常用）
#   "seurat_v3"   — 用于原始 counts（需配合 layers['counts']）
#   "cell_ranger" — 10x 官方方法
HVG_FLAVOR    = "seurat"


In [ ]:
# ============================================================
# 环境设置 —— 确保框架 src/ 可导入，然后一次性加载所有依赖
# ============================================================
# sys.path 必须在框架 import 之前设置，否则找不到 scrna_integration 模块。
import sys, os

_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# --- 所有 import 集中在这里 ---
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
import datetime

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

print("Loading upstream:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")


## 第一步：保留原始 counts

标准化操作会改变 `adata.X` 中的原始计数数据。但下游方法（scVI、scANVI、
DESeq2、pseudobulk）需要原始整数计数来精准建模——因此先把 counts 拷贝到
`adata.layers['counts']` 妥善保存。

In [ ]:
# 将原始 counts 保留到 layers['counts']。
# 下游方法（scVI / scANVI / DESeq2 / pseudobulk）需要原始计数数据，
# 而非标准化后的数据。
print("Copying raw counts to adata.layers['counts']...")
adata.layers['counts'] = adata.X.copy()
print(f"layers keys: {list(adata.layers.keys())}")


## 第二步：文库大小标准化

将每个细胞的总 counts 缩放到统一值（默认 10,000），消除不同细胞间
测序深度的差异。这一步之后，基因表达量在细胞间可比——
不会因为某个细胞被测了更多 reads 而显得"高表达"。

**为什么是 10,000？** 这是 scRNA-seq 领域的事实标准（scanpy 默认值）。
10,000 在"保留计数精度"和"数值稳定性"之间取得平衡。

In [ ]:
# 文库大小标准化：将每个细胞的总 counts 缩放到 target_sum。
print(f"\nNormalizing total counts per cell (target_sum=10000)...")
sc.pp.normalize_total(adata, target_sum=1e4)
print(f"After normalize_total: X mean={adata.X.mean():.2f}, max={adata.X.max():.0f}")


## 图1：标准化前后——每个细胞总 counts 分布

**看什么**：标准化前，不同细胞的总 UMI 计数差异悬殊（左图）。
标准化后，所有细胞拉到同一尺度（右图），后续比较不再受测序深度干扰。

In [ ]:
# 图1：标准化前后 per-cell 总 counts 分布对比
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 标准化前：从 layers['counts'] 计算每个细胞的总 UMI
counts_before = np.array(adata.layers['counts'].sum(axis=1)).flatten()
axes[0].hist(counts_before, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(np.median(counts_before), color='red', linestyle='--',
                label=f'中位={np.median(counts_before):.0f}')
axes[0].set_xlabel('总 UMI 计数')
axes[0].set_ylabel('细胞数')
axes[0].set_title('标准化前\n各细胞总 counts 差异大')
axes[0].legend(fontsize=9)

# 标准化后：从当前 adata.X 计算（normalize_total 之后，尚未 log1p）
counts_after = np.array(adata.X.sum(axis=1)).flatten()
axes[1].hist(counts_after, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(np.median(counts_after), color='red', linestyle='--',
                label=f'中位={np.median(counts_after):.0f}')
axes[1].set_xlabel('标准化后 counts')
axes[1].set_ylabel('细胞数')
axes[1].set_title('标准化后（target_sum=10000）\n所有细胞拉到同一尺度')
axes[1].legend(fontsize=9)

plt.suptitle('文库大小标准化效果', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/03_normalize_before_after.png', dpi=150, bbox_inches='tight')
plt.show()


## 第三步：Log1p 方差稳定化

标准化后的表达量仍然呈高度右偏分布——少数基因在少数细胞中表达极高，
大多数基因表达量接近于零。这种极端离散不适合 PCA 等线性降维方法。

`log(1+x)` 变换将数据拉近正态分布，使方差在不同表达水平之间更均匀
（方差稳定化）。加 1 是为了避免 `log(0)` 的数学问题——
单细胞数据中有大量零值。

In [ ]:
# Log1p 变换：log(1+x) 做方差稳定化，使数据适合 PCA 等线性方法。
print("\nApplying log1p transform...")
sc.pp.log1p(adata)
print(f"After log1p: X mean={adata.X.mean():.4f}, max={adata.X.max():.4f}")


## 图2：Log1p 前后——高表达基因分布变化

**看什么**：选取原始 counts 均值最高的 3 个基因，对比 Log1p 前后的分布形态。
左图（变换前）极度右偏，少数高值决定视觉尺度；
右图（变换后）接近正态，适合下游线性分析方法。

In [ ]:
# 图2：Log1p 前后——选取原始 counts 均值最高的 3 个基因对比分布
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 基于原始 counts 均值排名前 3 的基因
top_idx = np.argsort(
    np.array(adata.layers['counts'].mean(axis=0)).flatten()
)[-3:]
top_genes = adata.var_names[top_idx].tolist()
print(f"展示基因: {top_genes}")

# 手动从 layers['counts'] 重新计算标准化值（不取 log），作为 pre-log1p 对照
raw_top = adata.layers['counts'][:, top_idx].toarray()
lib_size = np.array(adata.layers['counts'].sum(axis=1)).flatten()
norm_expr = raw_top / lib_size[:, None] * 1e4  # 与 normalize_total 完全一致

# post-log1p 直接读取当前 adata.X
log_expr = adata.X[:, top_idx].toarray()

# 左图：pre-log1p（标准化后、未取 log）
for i, gene in enumerate(top_genes):
    axes[0].hist(norm_expr[:, i], bins=50, alpha=0.5, label=gene, density=True)
axes[0].set_xlabel('标准化后表达量（未 log）')
axes[0].set_ylabel('密度')
axes[0].set_title('Log1p 前：高度右偏\n少数细胞极高值主导')
axes[0].legend(fontsize=8)

# 右图：post-log1p
for i, gene in enumerate(top_genes):
    axes[1].hist(log_expr[:, i], bins=50, alpha=0.5, label=gene, density=True)
axes[1].set_xlabel('log1p 表达量')
axes[1].set_ylabel('密度')
axes[1].set_title('Log1p 后：接近正态\n适合 PCA 等线性方法')
axes[1].legend(fontsize=8)

plt.suptitle('Log1p 方差稳定化效果', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/03_log1p_before_after.png', dpi=150, bbox_inches='tight')
plt.show()


## 第四步：高可变基因 HVG 选择

大多数基因在所有细胞中表达量相近（管家基因），不携带区分细胞类型的信息。
高可变基因（HVG）是那些在不同细胞间表达差异最大的基因——
它们携带了细胞类型、状态差异的核心信号。

**为什么默认 2000 个？** 大量实证研究得出的平衡点：2000 个 HVG 通常已包含
几乎所有有生物学意义的变异信号，同时将数据维度从 ~20,000 压缩到 2,000，
大幅降噪、节省内存和计算时间。

**HVG 选择方法**：
- `seurat`：在 log-normalized 数据上计算（最常用，默认）
- `seurat_v3`：在原始 counts 上计算（适合极稀疏数据）
- `cell_ranger`：10x Genomics 官方方法

In [ ]:
# 高可变基因选择——识别变化最大的基因用于降维。
print(f"\nSelecting top {N_TOP_GENES} HVGs (flavor={HVG_FLAVOR})...")
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_TOP_GENES,
    flavor=HVG_FLAVOR,
)
n_hvg = adata.var['highly_variable'].sum()
print(f"HVGs selected: {n_hvg}")


## 图3：HVG 诊断图——平均表达量 vs 离散度

**看什么**：每个点是一个基因。X 轴是平均表达量（log scale），
Y 轴是标准化离散度。蓝色点是选中的高可变基因，灰色点是未选中的。
理想情况下，蓝色点应在各个表达水平上均匀覆盖高离散度区域——
如果集中在某个表达水平，说明 HVG 选择可能偏差。

In [ ]:
# 图3：HVG 诊断图——平均表达量 vs 标准化离散度
sc.pl.highly_variable_genes(adata, show=True)
plt.savefig('results/figures/03_hvg_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()
print("HVG 诊断图已保存至 results/figures/03_hvg_diagnostic.png")


## 图4：HVG 选择结果总览

**看什么**：从多少个基因中选出了多少个 HVG，占比多少。
如果选出的 HVG 数量明显少于预期（如设了 2000 但只得到 1,500），
可能是数据中高可变信号不足，或 flavor 参数与数据不匹配。

In [ ]:
# 图4：HVG 选择结果一览
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['全部基因', '高可变基因 HVG'], [adata.n_vars, n_hvg],
       color=['lightgray', 'steelblue'], edgecolor='white', width=0.4)
ax.set_ylabel('基因数')
ax.set_title(f'HVG 选择结果（{HVG_FLAVOR}, 目标 {N_TOP_GENES} 个）')
for i, val in enumerate([adata.n_vars, n_hvg]):
    ax.text(i, val + max(adata.n_vars, n_hvg) * 0.02,
            f'{val:,}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/03_hvg_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"总基因: {adata.n_vars:,}  ->  HVG: {n_hvg:,}  ({n_hvg/adata.n_vars*100:.1f}%)")


## 第五步：Float32 精度转换

单细胞标准化表达量和嵌入向量的有效信息远低于 float64 的 15 位有效数字。
转为 float32 后内存占用减半，对分析结果无实质影响——
这是单细胞数据分析的标准做法。

In [ ]:
# 转为 float32——内存减半，分析精度无实质影响。
print("\nCasting adata.X to float32...")
adata.X = adata.X.astype(np.float32)
print(f"X dtype now: {adata.X.dtype}")


## 第六步：记录参数、内存检查、写入 checkpoint

将标准化参数写入 `adata.uns` 以便追溯，执行内存自检确保 X 仍是稀疏 float32，
然后将产出写入磁盘形成 03 checkpoint。

In [ ]:
# 记录标准化参数——版本化键名（normalize_v1）允许不同参数多次重跑并存。
adata.uns['normalize_v1'] = {
    "target_sum":  1e4,
    "log_transformed": True,
    "hvg_flavor":  HVG_FLAVOR,
    "n_top_genes": N_TOP_GENES,
    "n_hvg":       n_hvg,
    "timestamp":   datetime.datetime.now().isoformat(),
}

# 记录 counts layer 的来源信息，便于追溯。
adata.uns["counts_layer"] = {
    "name": "counts",
    "description": "Raw counts preserved from 02 post-filter, before normalize_total",
    "source": UPSTREAM_PATH,
}

# 统一追踪字段——供迭代回跑追溯链使用。
adata.uns["stage"] = "03_normalized"
adata.uns["status"] = "experimental"          # PI 审查后手动改为 "promoted"
adata.uns["upstream"] = [UPSTREAM_PATH]       # 上游文件，完整溯源链
adata.uns["version"] = "v1"                   # 与 OUTPUT_PATH 版本号一致
print("normalize_v1:", adata.uns['normalize_v1'])

# --- 内存自检：确保 X 仍是稀疏 float32 ---
# 如果此处失败，说明上游某步不当 densify 或改了 dtype。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X invariants violated: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("Memory self-check passed: X is sparse CSR float32.")

# --- 写入 checkpoint ---
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"Wrote {OUTPUT_PATH}")

# 校验文件正确写出
assert os.path.exists(OUTPUT_PATH), f"Output NOT found: {OUTPUT_PATH}"
print(f"Verified: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")


## 第七步：释放内存

在同一 kernel 会话中继续跑 04 嵌入时，如果不释放当前 AnnData，
两个 stage 的数据会叠加占用内存资源。

In [ ]:
# 跨 stage 边界释放内存，避免在同一 kernel 中累积。
del adata
import gc
gc.collect()
print("Memory released.")
